# 저작권 RAG — 오프라인 검색과 평가 읽기

새 검색·평가 코어는 API·모델 다운로드 없이 실행합니다. 이 노트북은 공개 집계만 읽습니다.
문자 TF-IDF는 신경망 임베딩이 아니며, 기존 유료 임베딩 실험과 직접 비교하지 않습니다.
원문 검색·출처 일치가 법률적 정답이나 답변 가능성을 보장하지 않습니다.


In [ ]:
import json
from pathlib import Path
from html import escape
from IPython.display import display, HTML
ROOT = Path.cwd()
def report(name):
    return json.loads((ROOT/'reports'/name).read_text())
def show(rows):
    if not rows:
        return
    columns = list(rows[0])
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(row.get(c)))+'</td>' for c in columns)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))
def methods(data):
    rows=[]
    for item in data['methods']:
        quality = item['relevance_metrics'] or {}
        rows.append({'method':item['method'], 'Hit@3':quality.get('hit_at_k'),
                     'MRR@3':quality.get('mrr_at_k'), 'Recall@3':quality.get('recall_at_k'),
                     '중복률':item['avg_duplicate_rate'], '정답 평가 문항':item['relevance_scored_questions'],
                     '미검수':item['unjudged_questions_excluded_from_quality_scores'],
                     '근거 없음 질문에 결과 반환':item['unanswerable_with_any_result_rate']})
    show(rows)


## 1. 원본을 삭제하지 않고 사건과 기록 구분

원본 필수 5열이 비어 있는 행을 제외합니다. 같은 사건번호라도 다른 기록을 별도 record_id로 보존합니다.
해시 ID는 원문·메타데이터 기록을 구분하는 식별자이며 사건의 법적 동일성을 자동 보장하지 않습니다.


In [ ]:
quality = report('course-import-audit.json')
show([quality])


## 2. 수제 가상 예제로 기능 확인

8개 가상 기록으로 만든 toy fixture입니다. dev 3개와 test 10개를 분리했고 test 중 7개만 관련 기록 평가에 포함됩니다.
2개는 코퍼스에 필요한 근거가 없고 1개는 미검수입니다. 높은 Hit 점수는 실제 판례 검색 성능이 아닙니다.
문자 일부만 겹쳐도 검색 결과가 나오는 오탐도 함께 확인합니다.


In [ ]:
demo = report('demo-test-v1.json')
methods(demo)
assert demo['quality_scope'] == 'synthetic_fixture_only'


## 3. 기존 개발용 5문항은 독립 정답이 아님

과거 노트북의 기대 사건번호를 사용한 후향적 sanity check입니다. 추가 전문가 검수가 없고 문항 수도 작습니다.
방법별 차이를 관찰하는 데만 사용하며, 실제 독립 성능 향상이나 기존 임베딩 대비 우수성으로 소개하지 않습니다.


In [ ]:
legacy = report('course-legacy-v1.json')
methods(legacy)
show(legacy['mrr_changes_vs_body_bm25'])


## 4. 공식 12문항은 정답이 없어 품질 점수가 null

질문 파일에는 정답 판례가 없습니다. 미검수 질문을 모두 실패(0점)로 처리하지 않습니다.
검색 건수·중복만 관찰하고, 후보 원문이 포함된 로컬 검수 큐에서 검토자가 정답과 근거를 작성해야 합니다.
후보 자체를 자동으로 정답으로 만드는 방식은 사용하지 않습니다.


In [ ]:
official = report('course-official-unjudged-v1.json')
methods(official)
assert all(row['relevance_metrics'] is None for row in official['methods'])


## 5. 이번에 검증한 것

- 출처의 record_id와 문자열 위치가 원문에 일치하는지 검사
- 검색 결과에 같은 사건이 여러 번 나와도 정답 gain을 중복 부여하지 않음
- 검수 전 질문·근거 없음·정답 있음의 분모를 분리
- 같은 질문이 dev/test에 중복되면 오류로 표시
- 외부 패키지와 네트워크 없이 검색·평가 실행

출처 문자열 일치는 관련성·사실성 검증과 다릅니다. LLM 답변 생성·Judge는 이번에 실행하지 않았습니다.


In [ ]:
show([report('verification.json')])


## 재현

저장소 루트에서 README의 `python -m retrieval evaluate` 명령을 사용합니다.
원본 Excel·변환 JSON·검수 큐·검색 원문·문항별 결과는 `data/`, `artifacts/`에만 보관합니다.
지표를 올리기 위해 test 결과를 보고 설정을 바꾸면 새 독립 질문이 필요합니다.
